# CatBoost Baseline

This notebook trains a **CatBoostClassifier** as the first baseline for the smartphone addiction binary-classification competition. The full pipeline covers data loading, preprocessing, model training with early stopping, validation evaluation, feature importance analysis, and final submission generation.

---

## Table of Contents

1. [Setup and Data Loading](#baseline-setup-and-data-loading)
   - 1.1 [Imports](#baseline-imports)
   - 1.2 [Load Datasets](#baseline-load-datasets)
2. [Preprocessing](#baseline-preprocessing)
3. [Train / Validation Split](#baseline-train-validation-split)
4. [Model Training & Validation](#baseline-model-training-validation)
5. [Feature Importance](#baseline-feature-importance)
6. [Final Model & Submission](#baseline-final-model-submission)
   - 6.1 [Generate Predictions](#baseline-generate-predictions)
   - 6.2 [Build and Save Submission](#baseline-build-submission)
7. [Baseline Summary](#baseline-summary)

---

<a id="baseline-setup-and-data-loading" name="baseline-setup-and-data-loading"></a>

## 1. Setup and Data Loading

<a id="baseline-imports" name="baseline-imports"></a>

### 1.1 Imports

In [1]:
from pathlib import Path

import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

<a id="baseline-load-datasets" name="baseline-load-datasets"></a>
### 1.2 Load Datasets

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (691369, 14)
Test shape: (296302, 13)


<a id="baseline-preprocessing" name="baseline-preprocessing"></a>

## 2. Preprocessing

Separate the feature matrix `X` from the target vector `y` by dropping `addicted_label` (the target) and `id` (excluded from the baseline as a row identifier). The same column removal is applied to the test set to produce `X_test`, keeping train and test feature spaces in sync.

CatBoost can encode categorical columns natively, but it still requires the column to be free of `NaN` values when `cat_features` is passed to `fit()`. Replacing `NaN` with the explicit string `"Missing"` treats missingness as a distinct category rather than dropping that information entirely.

In [3]:
X = train.drop(columns=["addicted_label", "id"]).copy()
y = train["addicted_label"].copy()

X_test = test.drop(columns=["id"]).copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (691369, 12)
y: (691369,)
X_test: (296302, 12)


In [4]:
cat_cols = [
    "gender",
    "stress_level",
    "academic_work_impact",
]

for col in cat_cols:
    X[col] = X[col].fillna("Missing")
    X_test[col] = X_test[col].fillna("Missing")

<a id="baseline-train-validation-split" name="baseline-train-validation-split"></a>

## 3. Train / Validation Split

Split the labelled data 80 / 20 into a training set and a hold-out validation set.

- **`test_size=0.2`** — 20 % of rows are reserved for validation.
- **`random_state=42`** — fixes the shuffle for reproducibility.
- **`stratify=y`** — ensures both subsets preserve the original class ratio (~71 % addicted / ~29 % not addicted). Without stratification, an unlucky shuffle could under-represent one class in the validation fold, making the AUC estimate unreliable. This is especially important for imbalanced classification.

`train_test_split` always returns results in the fixed order: `X_train, X_valid, y_train, y_valid` (feature arrays first, label arrays second).

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

print()
print("Train target ratio:",y_train.value_counts(normalize=True),"\n")
print("Valid target ratio:", y_valid.value_counts(normalize=True),"\n")


Train: (553095, 12)
Valid: (138274, 12)

Train target ratio: addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64 

Valid target ratio: addicted_label
1    0.709425
0    0.290575
Name: proportion, dtype: float64 



<a id="baseline-model-training-validation" name="baseline-model-training-validation"></a>

## 4. Model Training & Validation

Train a `CatBoostClassifier` on `X_train` while monitoring validation AUC on `X_valid` after each boosting round.

| Parameter | Value | Notes |
|---|---|---|
| `iterations` | 500 | Maximum number of boosting rounds |
| `learning_rate` | 0.05 | Step size per boosting round |
| `depth` | 6 | Tree depth |
| `loss_function` | `Logloss` | Binary classification loss |
| `eval_metric` | `AUC` | Metric monitored on the validation set |
| `early_stopping_rounds` | 50 | Stop if validation AUC does not improve for 50 consecutive rounds |

`eval_set=(X_valid, y_valid)` provides the hold-out validation set used to monitor AUC during training, while `early_stopping_rounds=50` controls when training stops.

After training, evaluate the model on the validation set using ROC-AUC. `predict_proba(...)[:, 1]` returns the predicted probability of the positive class (`addicted_label = 1`).

In [6]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
)

0:	test: 0.9031016	best: 0.9031016 (0)	total: 156ms	remaining: 1m 17s
50:	test: 0.9291675	best: 0.9291675 (50)	total: 4s	remaining: 35.3s
100:	test: 0.9350226	best: 0.9350226 (100)	total: 8.08s	remaining: 31.9s
150:	test: 0.9384177	best: 0.9384177 (150)	total: 12.2s	remaining: 28.3s
200:	test: 0.9411362	best: 0.9411362 (200)	total: 17s	remaining: 25.2s
250:	test: 0.9433158	best: 0.9433158 (250)	total: 21.3s	remaining: 21.2s
300:	test: 0.9454591	best: 0.9454591 (300)	total: 25.6s	remaining: 16.9s
350:	test: 0.9470115	best: 0.9470115 (350)	total: 29.7s	remaining: 12.6s
400:	test: 0.9484022	best: 0.9484022 (400)	total: 34s	remaining: 8.38s
450:	test: 0.9497764	best: 0.9497764 (450)	total: 38.1s	remaining: 4.14s
499:	test: 0.9509171	best: 0.9509171 (499)	total: 42.2s	remaining: 0us

bestTest = 0.9509171423
bestIteration = 499



CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

In [7]:
valid_pred = model.predict_proba(X_valid)[:, 1]

valid_auc = roc_auc_score(
    y_valid,
    valid_pred
)

print(f"Validation AUC: {valid_auc:.6f}")

Validation AUC: 0.950917


<a id="baseline-feature-importance" name="baseline-feature-importance"></a>

## 5. Feature Importance

CatBoost exposes `feature_importances_` after training, providing a relative measure of how much each feature contributes to this model.

Screen-time-related features dominate the baseline model's feature importance, while the three categorical features receive zero importance in this model. These values describe the current CatBoost baseline only and should not be interpreted as causal effects or proof that a feature has no predictive value.

In [8]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance
1,daily_screen_time_hours,30.385165
8,weekend_screen_time,27.528522
2,social_media_hours,18.204339
7,app_opens_per_day,9.355938
6,notifications_per_day,8.380489
4,work_study_hours,2.784184
3,gaming_hours,1.922442
0,age,0.882247
5,sleep_hours,0.556674
9,gender,0.000000


<a id="baseline-final-model-submission" name="baseline-final-model-submission"></a>

## 6. Final Model & Submission

The search model above was trained on 80% of the labelled data, with the remaining 20% reserved for validation.
Use the best observed iteration count from validation to retrain the model on the full dataset `(X, y)`, allowing the final model to use all available training data.

The same hyperparameters are retained. `eval_set` and `early_stopping_rounds` are omitted because the iteration count has already been determined during validation.Finally, generate prediction probabilities for the test set and create the submission file using the `sample_submission` template.

In [9]:
best_iterations = model.get_best_iteration() + 1  # Add 1 because CatBoost uses 0-based indexing for iterations
print("Best iterations:", best_iterations)

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
)

final_model.fit(
    X,
    y,
    cat_features=cat_cols,
)

Best iterations: 500
0:	total: 81.9ms	remaining: 40.9s
50:	total: 4.95s	remaining: 43.6s
100:	total: 9.92s	remaining: 39.2s
150:	total: 14.5s	remaining: 33.4s
200:	total: 19.1s	remaining: 28.5s
250:	total: 23.9s	remaining: 23.7s
300:	total: 28.8s	remaining: 19.1s
350:	total: 33.7s	remaining: 14.3s
400:	total: 38.7s	remaining: 9.55s
450:	total: 43.4s	remaining: 4.72s
499:	total: 48.1s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

<a id="baseline-generate-predictions" name="baseline-generate-predictions"></a>

### 6.1 Generate Predictions

In [10]:
test_pred = final_model.predict_proba(X_test)[:, 1]

print(test_pred[:10])
print()
print("Min:", test_pred.min())
print("Max:", test_pred.max())
print("Mean:", test_pred.mean())

[0.99760025 0.91114701 0.96064097 0.97951233 0.99627315 0.80945235
 0.89952116 0.37776546 0.97295637 0.44671003]

Min: 0.0017004034841042914
Max: 0.9999913314126209
Mean: 0.7090278257274583


<a id="baseline-build-submission" name="baseline-build-submission"></a>

### 6.2 Build and Save Submission

Copy the `sample_submission` template (which already carries the correct `id` column) and overwrite `addicted_label` with the predicted probabilities. Verifying `isna().sum()` confirms no test row was silently dropped.

In [11]:
submission = sample_submission.copy()

submission["addicted_label"] = test_pred

display(submission.head())
print(submission.shape)
print(submission.isna().sum())

,id,addicted_label
0,691369,0.997600
1,691370,0.911147
2,691371,0.960641
3,691372,0.979512
4,691373,0.996273


(296302, 2)
id                0
addicted_label    0
dtype: int64


In [ ]:
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)  # Create the directory if it doesn't exist

submission_path = (
    SUBMISSION_DIR /
    "catboost_v1_baseline.csv"
)

submission.to_csv(
    submission_path,
    index=False,
)

print("Saved to:", submission_path)

Saved to: /Users/c.c./Developer/Competitions/predicting-smartphone-addiction/submissions/catboost_baseline_v1.csv


<a id="baseline-summary" name="baseline-summary"></a>

## 7. Baseline Summary

| Metric | Result |
|---|---:|
| Validation AUC | **0.950917** |
| Public LB | **0.95227** |
| Best iteration | **500 / 500** |

### Key findings

- The validation and Public LB scores differ by only **0.00135**, suggesting that the hold-out result is a credible first estimate.
- Daily screen time, weekend screen time, and social media hours contribute about **76.1%** of total feature importance, making screen-use behaviour the dominant signal in this baseline.
- The best iteration landed at the 500-round limit, so the model was still improving when training stopped.

![Baseline v1 Public Leaderboard](../images/baseline_v1_public_lb.png)

*Public leaderboard result for `catboost_baseline_v1.csv`.*

### Next direction

Use stratified 5-fold CV for a more reliable estimate, then raise the iteration ceiling with early stopping before moving to feature engineering.